# Career-ending injuries: logistic GLM

**What it predicts:** the probability that an injury ends a soccer player's professional career.

**Career-ending** means no professional appearance in any later season and no return within the injury's season (first such injury per player; at least two complete follow-up seasons required).

**Model:** logistic GLM (binomial, logit link), using only information known on the injury date.

```
logit P(career-ending) = body region + injury type + position + age + age^2
                         + log(1 + prior injuries) + re-injury within 60 days + prior injury to the same body part
                         + log(1 + games in last 12 months) + no games in last 12 months
                         + log(1 + career games) + year
```

**Headline result (held-out players):** AUC of **0.857**; the riskiest 10% of injuries by predicted probability capture **55%** of the career-ending injuries.

In [1]:
import numpy as np
import pandas as pd
import career

data = career.build_dataset()
print(f"{len(data):,} injuries; {data.career_ending.sum():,} career-ending ({data.career_ending.mean():.2%})")

120,682 injuries; 504 career-ending (0.42%)


## Held-out evaluation

Players are split 80/20 (each player's injuries all land on one side). The GLM is compared with a base rate that gives every injury the same probability. Higher `auc`, `avg_precision` and `top10_capture` are better; lower `log_loss` and `brier` are better.

In [2]:
train, test = career.split(data)
career.compare(career.fit_all(train), test)

  fitting Base rate, no predictors ...


  fitting Logistic GLM, main effects ...


,auc,avg_precision,log_loss,brier,mean_pred,top10_capture
model,,,,,,
"Base rate, no predictors",0.50000,0.00457,0.02921,0.00455,0.00408,0.00000
"Logistic GLM, main effects",0.85745,0.04972,0.02439,0.00447,0.00404,0.55046
(actual),NaN,NaN,NaN,NaN,0.00457,NaN


## Final fit

Refit on all injuries for the coefficients below. Regions and injury types with fewer than 20 career-ending injuries are pooled as **Other**.

In [3]:
glm = career.LogisticGLM(career.RHS_MAIN).fit(data)

## Coefficients

**How to read:** `odds_ratio` = exp(coef) multiplies the odds that an injury is career-ending. For example, 2.0 means twice the odds and 0.5 means half, holding the other predictors fixed. Reference: hamstring, unspecified injury type, midfielder, age 26, year 2015. Within each categorical variable, levels are sorted from highest to lowest odds ratio.

In [4]:
coefs = career.coefficients(glm)
coefs.insert(1, "odds_ratio", np.exp(coefs["coef"]).round(3))
group = coefs.index.str.split(" = ").str[0]
order = pd.Series(range(len(group)), index=group).groupby(level=0).min()
coefs = (coefs.assign(_g=group.map(order), _s=-coefs["odds_ratio"])
         .sort_values(["_g", "_s"]).drop(columns=["_g", "_s"]))
coefs

,coef,odds_ratio,p_value
variable,,,
Intercept,-4.6377,0.010,0.0000
region_g = Achilles,2.3904,10.918,0.0000
region_g = Knee,2.3896,10.909,0.0000
region_g = Unknown,1.4244,4.155,0.0053
region_g = Other,0.6327,1.883,0.2177
nature_g = Surgery,1.1935,3.299,0.0000
nature_g = Tear / rupture,1.0771,2.936,0.0000
nature_g = Fracture,0.8878,2.430,0.0001
nature_g = Ligament / joint,0.4423,1.556,0.0154


## More

- [MODEL_SUMMARY.md](MODEL_SUMMARY.md): plain-language summary of both models, with worked examples.
- [README.md](README.md): setup, data sources, and how to run the scripts.
- Code: [severity.py](severity.py) and [career.py](career.py).